
Generates ~1000 small JSON files simulating an incremental data feed:
  - ~950 "normal" files: standard schema, consumption_barrels as int
  - ~40 "evolved" files: adds a new field `price_per_barrel` (schema evolution demo)
  - ~10 "widened" files: consumption_barrels exceeds Int32 range (type widening demo)


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev_ua5816bd.roksolana_shendiu770.bronze_landing;

In [0]:
import json
import random
import uuid
import os
from datetime import datetime, timedelta

OUTPUT_DIR = "/Volumes/dbr_dev_ua5816bd/roksolana_shendiu770/bronze_landing/petroleum_consumption"
NUM_NORMAL_FILES = 950
NUM_EVOLVED_FILES = 40      
NUM_WIDENED_FILES = 10     
RECORDS_PER_FILE_MIN = 10
RECORDS_PER_FILE_MAX = 40

PADD_REGIONS = ["PADD 1", "PADD 2", "PADD 3", "PADD 4", "PADD 5"]
PRODUCTS = ["gasoline", "diesel", "jet_fuel", "heating_oil"]

INT32_MAX = 2_147_483_647


def random_period(days_back_max=1800):
    d = datetime(2026, 8, 24) - timedelta(days=random.randint(0, days_back_max))
    return d.strftime("%Y-%m-%d")


def make_record(evolved=False, widened=False):
    record = {
        "record_id": str(uuid.uuid4()),
        "period": random_period(),
        "padd_region": random.choice(PADD_REGIONS),
        "product": random.choice(PRODUCTS),
        "consumption_barrels": random.randint(500, 250_000),
        "unit": "thousand barrels",
        "source_system": "synthetic_sensor_feed",
    }
    if widened:
        record["consumption_barrels"] = INT32_MAX + random.randint(1, 5_000_000)
    if evolved:
        record["price_per_barrel"] = round(random.uniform(55.0, 95.0), 2)
    return record


def write_file(path, records):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


def generate_batch(prefix, count, evolved=False, widened=False, start_index=0):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for i in range(count):
        file_index = start_index + i
        n_records = random.randint(RECORDS_PER_FILE_MIN, RECORDS_PER_FILE_MAX)
        records = [make_record(evolved=evolved, widened=widened) for _ in range(n_records)]
        file_name = f"{prefix}_{file_index:05d}.json"
        write_file(f"{OUTPUT_DIR}/{file_name}", records)
    print(f"Wrote {count} '{prefix}' files ({'evolved' if evolved else 'normal'}{' + widened' if widened else ''}) to {OUTPUT_DIR}")



In [0]:
random.seed(42)
generate_batch("part", NUM_NORMAL_FILES, evolved=False, widened=False, start_index=0)

In [0]:
generate_batch("part_evolved", NUM_EVOLVED_FILES, evolved=True, widened=False,
                start_index=NUM_NORMAL_FILES)

In [0]:
def make_renamed_record():
    return {
        "record_id": str(uuid.uuid4()),
        "period": random_period(),
        "padd_region": random.choice(PADD_REGIONS),
        "product": random.choice(PRODUCTS),
        "consumption_volume_barrels": random.randint(500, 250_000),  
        "unit": "thousand barrels",
        "source_system": "synthetic_sensor_feed",
        "price_per_barrel": round(random.uniform(55.0, 95.0), 2),
    }

def generate_renamed_batch(prefix, count, start_index):
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for i in range(count):
        file_index = start_index + i
        n_records = random.randint(RECORDS_PER_FILE_MIN, RECORDS_PER_FILE_MAX)
        records = [make_renamed_record() for _ in range(n_records)]
        file_name = f"{prefix}_{file_index:05d}.json"
        write_file(f"{OUTPUT_DIR}/{file_name}", records)
    print(f"Wrote {count} '{prefix}' files (renamed field) to {OUTPUT_DIR}")

generate_renamed_batch("part_renamed", 10, start_index=1000)

In [0]:
generate_batch("part_trigger_test1", 15, evolved=False, widened=False,
                start_index=2000)